In [11]:
import pandas as pd
from pathlib import Path

clean = Path("/Users/trivenidhamdhere/Documents/MSc Data Science/"
             "MSc Dissertation - AZ/cleaned_track_data")

# load every parquet
tables = {f.stem: pd.read_parquet(f, engine="fastparquet")
          for f in sorted(clean.glob("*.parquet"))}

# lowercase headers + all string data
def lower_all(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype("string").str.strip().str.lower()
    return df
tables = {name: lower_all(df) for name, df in tables.items()}

# unique model_id set per table (None if the table has no model_id column)
def model_ids(df):
    col = next((c for c in df.columns
                if c in ("model_id", "depmap_id", "modelid", "ach_id")), None)
    return set(df[col].dropna().unique()) if col else None

id_sets  = {name: model_ids(df) for name, df in tables.items()}
have_id  = {name: s for name, s in id_sets.items() if s}
no_id    = [name for name, s in id_sets.items() if s is None]

In [12]:
# build coverage over the union of all model_ids
all_ids  = sorted(set().union(*have_id.values()))
coverage = pd.DataFrame({"model_id": all_ids})
for name, s in have_id.items():
    coverage[name] = coverage["model_id"].isin(s)
coverage["n_modalities"] = coverage[list(have_id)].sum(axis=1)

# --- show output ---
print("tables WITH model_id:", list(have_id))
print("tables WITHOUT model_id (excluded):", no_id)
print(f"\ndistinct model_ids: {len(coverage)}")
print("\nper-modality counts:")
print(coverage[list(have_id)].sum().sort_values(ascending=False))
print(f"\nlines present in ALL {len(have_id)} modalities:",
      (coverage["n_modalities"] == len(have_id)).sum())
print("\nmodalities-per-line distribution:")
print(coverage["n_modalities"].value_counts().sort_index())

coverage

tables WITH model_id: ['geo_info', 'metabolomics', 'proteomics']
tables WITHOUT model_id (excluded): []

distinct model_ids: 1053

per-modality counts:
metabolomics    928
geo_info        588
proteomics      375
dtype: int64

lines present in ALL 3 modalities: 242

modalities-per-line distribution:
n_modalities
1    457
2    354
3    242
Name: count, dtype: int64


,model_id,geo_info,metabolomics,proteomics,n_modalities
0,ach-000001,True,True,True,3
1,ach-000002,True,True,False,2
2,ach-000003,True,False,False,1
3,ach-000004,False,True,True,2
4,ach-000005,True,True,True,3
...,...,...,...,...,...
1048,ach-002319,True,False,False,1
1049,ach-002327,True,False,False,1
1050,ach-002328,True,False,False,1
1051,ach-002345,True,False,False,1
